# Section 2.1

In [1]:
import pandas as pd
import numpy as np

path = "/Users/daniilg/Downloads/dfa_analysis_data.xlsx"
factors = pd.read_excel(path, sheet_name="factors")

# Date handling: adjust the column name if needed
date_col = factors.columns[0]
factors[date_col] = pd.to_datetime(factors[date_col].astype(str), format="mixed")
factors = factors.set_index(date_col).sort_index()
F = factors[["Mkt-RF", "SMB", "HML"]]

subsamples = {
    "Jul 1926 - Dec 1980": ("1926-07-01", "1980-12-31"),
    "Jan 1981 - Dec 2001": ("1981-01-01", "2001-12-31"),
    "Jan 2002 - Aug 2026": ("2002-01-01", "2026-08-31"),
}

def perf_stats(df):
    out = pd.DataFrame({
        "Ann. Mean": df.mean() * 12,
        "Ann. Vol": df.std() * np.sqrt(12),
    })
    out["Ann. Sharpe"] = out["Ann. Mean"] / out["Ann. Vol"]
    out["VaR(.05)"] = df.quantile(0.05)
    return out

# Section 1
stats = pd.concat(
    {name: perf_stats(F.loc[s:e]) for name, (s, e) in subsamples.items()},
    names=["Subsample", "Factor"],
)
stats


Ann. Mean  Ann. Vol  Ann. Sharpe  VaR(.05)
Subsample           Factor                                            
Jul 1926 - Dec 1980 Mkt-RF   0.080958  0.204988     0.394938 -0.084090
                    SMB      0.033949  0.114326     0.296947 -0.041875
                    HML      0.050242  0.134342     0.373989 -0.044375
Jan 1981 - Dec 2001 Mkt-RF   0.077848  0.157183     0.495269 -0.064135
                    SMB     -0.002019  0.117282    -0.017215 -0.045835
                    HML      0.064438  0.109823     0.586743 -0.041750
Jan 2002 - Aug 2026 Mkt-RF   0.093827  0.152528     0.615145 -0.076725
                    SMB      0.007500  0.087215     0.085994 -0.038850
                    HML      0.005530  0.106192     0.052073 -0.041375

# Section 2

In [2]:
premium = stats["Ann. Mean"].unstack("Factor") > 0
premium

perf_stats(F.loc["1990-01-01":"1999-12-31"])


,Ann. Mean,Ann. Vol,Ann. Sharpe,VaR(.05)
Mkt-RF,0.12792,0.138038,0.926698,-0.049440
SMB,-0.01548,0.102609,-0.150864,-0.051275
HML,0.00052,0.090248,0.005762,-0.039685


No, mkt-RF is great, HML also has positive annualized mean but notably Small minus Big does not have a positive annualized mean. 

Yes; the premium for SMB goes from + (pre 1980) to - (1980-2000ish) to + (2002 - 2026). It's positive value in the last period from 2002 to 2026 is around 0.007 compared to 0.03 from pre 1980 period which is smaller.

No, value premium actually goes up slightly in this period by + 0.01.

All 3 factors are positive for the most recent period but notably the only one that is at "good" as it was in the 1960s is mkt-RF e.g. holding stocks in general. The other two are a factor of 10x smaller than they were in the 1960s.

# Section3


In [3]:
corr_full = F.corr()
corr_sub = {name: F.loc[s:e].corr() for name, (s, e) in subsamples.items()}

corr_full
for name, c in corr_sub.items():
    display(c.rename_axis(name))

,Mkt-RF,SMB,HML
Jul 1926 - Dec 1980,,,
Mkt-RF,1.000000,0.373879,0.442576
SMB,0.373879,1.000000,0.313398
HML,0.442576,0.313398,1.000000


,Mkt-RF,SMB,HML
Jan 1981 - Dec 2001,,,
Mkt-RF,1.000000,0.158345,-0.530464
SMB,0.158345,1.000000,-0.444280
HML,-0.530464,-0.444280,1.000000


,Mkt-RF,SMB,HML
Jan 2002 - Aug 2026,,,
Mkt-RF,1.000000,0.303519,0.087385
SMB,0.303519,1.000000,0.123553
HML,0.087385,0.123553,1.000000


It depends on how you define small correlation between factors: the correlation in the first period is around 0.3 to 0.45 between factors which is medium-ish correlation, second period SMB and Mkt-Rf is 0.15 which is great but then mkt-RF and HML is -0.53 which is not as good. In the last period they actulaly do have the smallest porrelations of 0.3, 0.08, and 0.12. So it's not actually trivial to distil these factors from each other and in the first and second period I would argue the correlation isn't well distilled between them